# Validation of the Train and Test Files

**Objective:** Before any model is trained, check that the final train and test files are consistent with each other and that the test set can be used as an honest evaluation: no record is shared between them, and the test period comes after the training period.

**Files:** The checks use the files produced in the pre-processing stage: `1_8_2.csv` and `1_8_2_test.csv` (scaled) for the structural comparisons, and `1_8_1.csv` and `1_8_1_test.csv` (unscaled) to rebuild the purchase dates, which are not readable after scaling.

| Check | Question | How it is tested |
|---|---|---|
| 1 | Are there repeated rows inside each file? | Duplicates over all columns except `ID` and `random_noise` |
| 2 | Is any test row a copy of a train row? | Inner merge of test with train on the same columns |
| 3 | Do the two files share transaction `ID`s? | `ID` of test found in train |
| 4 | How many test customers already exist in train? | `customer_id` of test found in train |
| 5 | Does the test period come after the training period? | First and last purchase date of each file |

In [1]:
import pandas as pd

# Files from pre-processing: 1.8.2 (scaled) and 1.8.1 (unscaled, with the raw date parts)
trainFile = "../../data/preprocessing/1_8_2.csv"
testFile = "../../data/preprocessing/1_8_2_test.csv"

trainFileUnscaled = "../../data/preprocessing/1_8_1.csv"
testFileUnscaled = "../../data/preprocessing/1_8_1_test.csv"

# Load the scaled datasets for the structural comparisons
train = pd.read_csv(trainFile)
test = pd.read_csv(testFile)

# Leave the transaction ID and the random noise control out of the duplicate comparison
columns = [c for c in test.columns if c not in ["ID", "random_noise"]]

# 1. Repeated rows inside each file
print(f"repeated rows inside train: {train.duplicated(subset=columns).sum()}")
print(f"repeated rows inside test:  {test.duplicated(subset=columns).sum()}")

# 2. Test rows that are a copy of a train row
copies = test.merge(train[columns].drop_duplicates(), on=columns, how="inner")
print(f"test rows identical to a train row: {len(copies)}")

# 3. Check for same ID in both files
print(f"IDs in both files: {test['ID'].isin(train['ID']).sum()}")

# 4. Customers in both files
if "customer_id" in test.columns and "customer_id" in train.columns:
  in_train = test["customer_id"].isin(train["customer_id"])
  print(
      f"test customers also in train:"
      f" {test.loc[in_train, 'customer_id'].nunique()} of"
      f" {test['customer_id'].nunique()} ({in_train.sum()} of {len(test)} test"
      " rows)"
  )

# 5. Dates: Carregar atributos temporais não escalados para reconstrução correta
date_cols = ["purchase_year", "purchase_month", "purchase_day"]

try:
  # Read from the 1.8.1 files, where the date parts are still integers
  train_raw = pd.read_csv(trainFileUnscaled, usecols=date_cols)
  test_raw = pd.read_csv(testFileUnscaled, usecols=date_cols)

  train_dates = pd.to_datetime(
      train_raw.rename(
          columns={
              "purchase_year": "year",
              "purchase_month": "month",
              "purchase_day": "day",
          }
      )
  )
  test_dates = pd.to_datetime(
      test_raw.rename(
          columns={
              "purchase_year": "year",
              "purchase_month": "month",
              "purchase_day": "day",
          }
      )
  )

  print(
      f"train period: {train_dates.min().date()} to {train_dates.max().date()}"
  )
  print(
      f"test period:  {test_dates.min().date()} to {test_dates.max().date()}"
  )

except Exception as e:
  print(
      "\nNota: Não foi possível calcular o intervalo exacto de datas através do"
      " ficheiro escalado."
  )
  print(f"Mensagem de salvaguarda: {e}")


repeated rows inside train: 0
repeated rows inside test:  0
test rows identical to a train row: 0
IDs in both files: 0
test customers also in train: 184 of 206 (932 of 986 test rows)
train period: 2015-07-11 to 2025-02-15
test period:  2025-05-24 to 2026-10-06


**Conclusions:**
- **No repeated or shared records (checks 1 to 3):** there are no repeated rows in train (0) or in test (0), no test row is identical to a train row (0), and no `ID` appears in both files (0). `ID` and `random_noise` were left out of the comparison on purpose, so that two rows that differ only in these columns still count as the same transaction. The test set therefore has no exact copy of a training transaction and cannot inflate the results through duplicates.
- **Most test customers are known (check 4):** 184 of the 206 test customers (89%) also appear in train, and they account for 932 of the 986 test rows (94.5%). Only 22 customers (54 rows, 5.5%) are new. This matches the real use of the model, which predicts the next purchase of customers the company already knows, and it is why the history features (`prior_*`, `customer_tenure_days`) are informative. It also means that `customer_id` must stay out of the feature set, and that the performance measured on the test set mostly describes returning customers, not new ones.
- **Test comes after train (check 5):** the training transactions run from 2015-07-11 to 2025-02-15 and the test ones from 2025-05-24 to 2026-10-06, so the two periods do not overlap. The gap between them is 98 days. If the target `repeat_purchase_90d` looks 90 days ahead, as its name suggests, the outcome window of the last training transaction ends before the first test date, so no information from the test period enters the training labels.
- **Different periods, different behaviour:** train covers about 9.6 years and test about 1.6 years. Earlier steps already showed differences between the two sets (for example, the share of first purchases was 12.6% in train and 5.5% in test). The results on the test set should therefore be compared with the temporal validation used during development, and a drop in performance is not necessarily a sign of overfitting.

The files pass all the checks and are ready to be used for the final model and for the predictions.